## Educational Analytics Prediction Platform

### Data Cleaning

\Importing Necessary Libraries

In [1]:
import numpy as np
import pandas as pd
import os

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns",None)
pd.set_option("display.width",1000)

\Loading the dataset

In [2]:
df = pd.read_csv(r"C:\NG\Educational-Analytics-Platform\data\raw\student_dropout_dataset.csv")
df.head()

,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


\Missing Value Detection

In [15]:
def detect_missing_values(df):

    numeric_cols = df.select_dtypes(include=np.number).columns
    categorical_cols = df.select_dtypes(include='object').columns
    
    missing_counts = df.isnull().sum()
    missing_counts = missing_counts[missing_counts>0]

    details = {}
    missing_numeric = []
    missing_categorical = []
    
    for col, count in missing_counts.items():
        col_type = "numeric" if col in numeric_cols else "categorical"
        details[col] = {
            "count": int(count),
            "pct": round((count / len(df)) * 100, 2),
            "type": col_type,
        }
        if col_type == "numeric":
            missing_numeric.append(col)
        else:
            missing_categorical.append(col)

    report = {
        "has_missing": len(missing_counts) > 0,
        "total_missing": int(missing_counts.sum()),
        "details": details,
        "numeric_cols": missing_numeric,
        "categorical_cols": missing_categorical,
    }
    return report

In [19]:
missing_report = detect_missing_values(df)
missing_report

{'has_missing': True,
 'total_missing': 2011,
 'details': {'Family_Income': {'count': 500, 'pct': 5.0, 'type': 'numeric'},
  'Study_Hours_per_Day': {'count': 500, 'pct': 5.0, 'type': 'numeric'},
  'Stress_Index': {'count': 500, 'pct': 5.0, 'type': 'numeric'},
  'Parental_Education': {'count': 511, 'pct': 5.11, 'type': 'categorical'}},
 'numeric_cols': ['Family_Income', 'Study_Hours_per_Day', 'Stress_Index'],
 'categorical_cols': ['Parental_Education']}

\Missing Value Treatment

In [20]:
def treat_missing_values(df, threshold=0.50):
    df_clean = df.copy()

    numeric_cols = df_clean.select_dtypes(include=np.number).columns
    categorical_cols = df_clean.select_dtypes(include='object').columns

    fill_values = {}
    dropped_cols = []
    filled_log = []

    missing_counts = df_clean.isnull().sum()
    missing_cols = missing_counts[missing_counts > 0].index

    for col in missing_cols:
        missing_count = missing_counts[col]
        missing_ratio = missing_count / len(df_clean)

        if missing_ratio > threshold:
            df_clean = df_clean.drop(columns=[col])
            dropped_cols.append((col, round(missing_ratio * 100, 2)))
            continue

        if col in numeric_cols:
            value = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(value)
            fill_values[col] = value
            filled_log.append((col, "median", round(value, 2), int(missing_count)))

        elif col in categorical_cols:
            value = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(value)
            fill_values[col] = value
            filled_log.append((col, "mode", value, int(missing_count)))

    report = {
        "dropped": dropped_cols,
        "filled": filled_log,
        "missing_after": int(df_clean.isnull().sum().sum()),
    }
    return df_clean, fill_values, report

In [21]:
if missing_report["has_missing"]:
    df_clean, fill_values, treat_report = treat_missing_values(df)

In [22]:
treat_report

{'dropped': [],
 'filled': [('Family_Income', 'median', np.float64(29740.5), 500),
  ('Study_Hours_per_Day', 'median', np.float64(4.0), 500),
  ('Stress_Index', 'median', np.float64(5.5), 500),
  ('Parental_Education', 'mode', 'Bachelor', 511)],
 'missing_after': 0}

\Detecting Duplicate Records

In [23]:
def detect_duplicates(df):
   
    full_dup_count = int(df.duplicated().sum())
    id_like_columns = [
        col for col in df.columns
        if df[col].nunique() == len(df)
    ]

    report = {
        "full_duplicate_count": full_dup_count,
        "full_duplicate_pct": round((full_dup_count / len(df)) * 100, 2),
        "id_like_columns": id_like_columns,
    }
    return report

In [25]:
dup_report = detect_duplicates(df)

print("=== DUPLICATE DETECTION ===")
print(f"Full duplicate rows : {dup_report['full_duplicate_count']} "
      f"({dup_report['full_duplicate_pct']}%)")
print(f"ID-like columns     : {dup_report['id_like_columns']}")

=== DUPLICATE DETECTION ===
Full duplicate rows : 0 (0.0%)
ID-like columns     : ['Student_ID']


\Removing Duplicate

In [26]:
def remove_duplicates(df):

    rows_before = len(df)
    df_clean = df.drop_duplicates().reset_index(drop=True)
    rows_removed = rows_before - len(df_clean)
    return df_clean, rows_removed

In [28]:
df_clean, rows_removed = remove_duplicates(df)

print("\n=== DUPLICATE REMOVAL ===")
print(f"Shape before : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Shape after  : {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")
print(f"Rows removed : {rows_removed}")


=== DUPLICATE REMOVAL ===
Shape before : 10000 rows x 19 columns
Shape after  : 10000 rows x 19 columns
Rows removed : 0


\Outlier Detection

In [1]:
def detect_outliers(df, target=None, min_unique=10):
    
    numeric_cols = df.select_dtypes(include=np.number).columns

    report = {}
    for col in numeric_cols:

        if target is not None and col == target:
            continue
        
        if df[col].nunique() < min_unique:
            continue

        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        if iqr == 0:
            continue

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_mask = (df[col] < lower) | (df[col] > upper)
        count = int(outlier_mask.sum())

        if count > 0:
            report[col] = {
                "outlier_count": count,
                "outlier_pct": round((count / len(df)) * 100, 2),
                "lower_bound": round(lower, 2),
                "upper_bound": round(upper, 2),
            }
    return report

In [30]:
def treat_outliers(df, outlier_report):
    df_clean = df.copy()
    capped_log = {}

    for col, info in outlier_report.items():
        lower = info["lower_bound"]
        upper = info["upper_bound"]
        
        df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
        capped_log[col] = info["outlier_count"]

    return df_clean, capped_log